# 1. Introdução

&emsp; O presente notebook documenta a fase de análise exploratória de modelos, na qual dois algoritmos de *ensemble* de alta performance, **XGBoost** e **AdaBoost**, foram implementados e avaliados. É importante ressaltar que os modelos aqui apresentados **não constituem a solução preditiva final** do projeto, mas representam uma etapa investigativa fundamental. O objetivo deste estudo comparativo foi estabelecer modelos para comparação e, principalmente, gerar conclusões valiosas sobre a dinâmica dos dados e a natureza dos trade-offs de previsão em diferentes estágios do semestre. As conclusões e os aprendizados obtidos nesta análise foram cruciais para fundamentar a escolha e a subsequente avaliação do modelo definitivo, que é detalhado em um notebook separado.

## 1.1 Importação de bibliotecas

&emsp; Nesta célula, é feita a importação das principais bibliotecas e funções usadas tanto para o processamento quanto para os modelos testados.

In [ ]:
import re
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
import plotly.io as pio 
pio.renderers.default = "vscode"

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import recall_score, ConfusionMatrixDisplay, confusion_matrix, f1_score
from imblearn.over_sampling import SMOTE

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier

# Caminhos de entrada
XLSX1_PATH = "../../Datos_Anonimo_20231_v2.xlsx"  # 1º semestre
XLSX2_PATH = "../../Datos_Anonimo_20232_v2.xlsx"  # 2º semestre
SHEET_INDEX = 1  # segunda aba

## 1.2 Importando todas as tabelas/dados

&emsp; Os dados importados estão divididos em três dataframes devido às semanas que a previsão será realizada: `df1` representa a semana 6, `df2` representa a semana 8, `df3`representa a semana 12. Além disso, ocorre também a divisão em dataframes de treino e de teste, em que `df1`, `df2` e `df3` são os dados para treino, equanto `df1_test`, `df2_test` e `df3_test` são para teste.

In [ ]:
df1 = pd.read_csv('../dados/dados_modelo1.csv')
df2 = pd.read_csv('../dados/dados_modelo2.csv')
df3 = pd.read_csv('../dados/dados_modelo3.csv')
df1_test = pd.read_csv('../dados/dados_teste1.csv')
df2_test = pd.read_csv('../dados/dados_teste2.csv')
df3_test = pd.read_csv('../dados/dados_teste3.csv')

## 1.3 Separação da variável alvo nos 3 treinos e testes

&emsp; Nesta célula será feita a definição de qual variável irá ser prevista (exemplo: X1_train = df1.drop('Reprovou', axis = 1)) e qual feature não entrará no treinamento e no teste dos modelos escolhidos.

In [ ]:
X1_train = df1.drop('Reprovou', axis = 1)
y1_train = df1['Reprovou']

X2_train = df2.drop('Reprovou', axis = 1)
y2_train = df2['Reprovou']

X3_train = df3.drop('Reprovou', axis = 1)
y3_train = df3['Reprovou']

X1_test = df1_test.drop('Reprovou', axis = 1)
y1_test = df1_test['Reprovou']

X2_test = df2_test.drop('Reprovou', axis = 1)
y2_test = df2_test['Reprovou']

X3_test = df3_test.drop('Reprovou', axis = 1)
y3_test = df3_test['Reprovou']

## 1.4 Trocando Valores 

&emsp; Na coluna "TempoQ3", existe um dado que está Nulo, então aqui será feita a substituição para 0.

In [ ]:
X1_train['TempoQ3'] = X1_train['TempoQ3'].fillna(0)
X2_train['TempoQ3'] = X2_train['TempoQ3'].fillna(0)
X3_train['TempoQ3'] = X3_train['TempoQ3'].fillna(0)

# 2. Treinamento e testes dos modelos.

## 2.1 XGBoost

&emsp; O XGBoost (Extreme Gradient Boosting) é um algoritmo de aprendizado supervisionado baseado em árvores de decisão, voltado principalmente para problemas de classificação e regressão. Ele utiliza a técnica de boosting, em que vários modelos fracos (normalmente árvores de decisão rasas) são combinados de forma sequencial, e cada novo modelo busca corrigir os erros cometidos pelos anteriores.

&emsp; O termo “Gradient” refere-se ao fato de que a otimização é realizada por meio do gradiente da função de perda, garantindo que cada nova árvore seja ajustada na direção que minimiza os erros. Já o “Boosting” indica a estratégia de construção progressiva do modelo, onde o desempenho é aprimorado a cada iteração.

&emsp; Desse modo, o XGBoost foi utilizado no modelo preditivo atual para prever se os alunos irão reprovar, a partir de três momentos de análise (semanas 6, 9 e 12). As variáveis utilizadas como preditores foram as notas, o gênero e a informação se o aluno pertence ou não à área da disciplina, permitindo identificar padrões de desempenho ao longo do curso.

#### O Tratamento de Dados Desbalanceados com SMOTE

&emsp; Um dos principais desafios em problemas de classificação, especialmente na previsão de eventos raros como a reprovação, é o **desbalanceamento de classes**. Em nosso conjunto de dados, o número de alunos aprovados (classe majoritária) é significativamente maior que o de reprovados (classe minoritária). Se um modelo for treinado diretamente com esses dados, ele pode se tornar enviesado, aprendendo a prever principalmente a classe dominante e ignorando a classe de interesse que são os reprovados, resultando em um baixo poder preditivo para identificar os alunos em risco.

&emsp; Para mitigar esse problema, utilizamos a técnica **SMOTE (Synthetic Minority Over-sampling Technique)**. Diferente de uma simples duplicação dos dados da classe minoritária, o SMOTE gera **novos exemplos sintéticos** que são plausíveis e representativos dos alunos reprovados. O algoritmo funciona da seguinte forma:
1.  Seleciona um aluno da classe minoritária (reprovado).
2.  Identifica seus "vizinhos" mais próximos (outros alunos com características similares que também foram reprovados).
3.  Cria um novo dado sintético em um ponto aleatório no segmento de linha que conecta o aluno original a um de seus vizinhos.

&emsp; Essa abordagem cria um conjunto de dados de treino mais rico e balanceado, forçando o modelo a aprender as características que definem um aluno reprovado com mais atenção. É fundamental destacar que o SMOTE foi aplicado **exclusivamente no conjunto de dados de treino**, após a separação do conjunto de teste. Essa prática é crucial para evitar o **vazamento de dados (data leakage)**. Se o SMOTE fosse aplicado antes da divisão, informações sintéticas criadas a partir dos dados de treino poderiam "vazar" para o conjunto de teste, fazendo com que o modelo fosse avaliado em dados que ele, indiretamente, já viu. A aplicação correta garante que a avaliação de performance do modelo seja realista e fidedigna.

#### Explicação dos parâmetros usados

&emsp; Os parâmetros do XGBoost são características que controlam como o modelo é treinado e testado, influenciando diretamente sua capacidade de aprendizado e generalização. Em teoria, quanto mais parâmetros ajustamos, maior seria a possibilidade de encontrar uma configuração ideal. No entanto, em nosso projeto de previsão de reprovação dos alunos nas semanas 6, 9 e 12, observamos que essa lógica não se confirmou na prática. Cada vez que ampliávamos o conjunto de parâmetros a serem avaliados pelo GridSearch, os resultados do modelo pioravam em comparação à configuração inicial com parâmetros padrão. Um exemplo claro disso: colocando para ser analisado com parâmetros como "gamma", "reg_alpha" e "reg_lambda", principalmente na terceira análise, o modelo coloca **menos verdadeiros positivos** coloca **mais dados verdadeiros negativos**.

&emsp; A configuração que se mostrou mais adequada foi simples: xgb = XGBClassifier(random_state=42, eval_metric='logloss'), com o GridSearch testando apenas três parâmetros básicos:

- **n_estimators:** Número de árvores ideal encontrada pelo GridSearch foi 200 para a primeira e segunda análise e 100 para a última análise.
- **max_depth:** Profundidade máxima, com melhores valores encontrados pelo GridSearch foi 7 na primeira e segunda análise e 5 na última análise.
- **learning_rate:** Taxa de aprendizado, com os melhores valores encontrados pelo GridSearch foram 0,1 nas 3 análises.
- **random_state:** Semente aleatória para garantir reprodutibilidade. por recomendação de professores, colocamos o valor de 42.
- **tree_method:** Define o algoritmo usado para construir árvores. Para análises, tanto na documentação do site XGboost quanto em testes, foi observado que "auto", é o melhor parâmetro por ter datset pequeno, não é preciso mudar esse parâmetro.

&emsp; Essa escolha se mostrou suficiente para ajustar o modelo sem comprometer seu desempenho, garantindo resultados mais consistentes do que quando tentávamos incluir parâmetros adicionais como gamma, subsample ou regularizações.

&emsp; Isso mostra que, em problemas como o nosso — onde as features são apenas notas, gênero e se o aluno pertence ou não à área da matéria — a simplicidade pode ser mais eficaz do que um ajuste excessivamente detalhado. Em vez de melhorar, o excesso de parâmetros fez com que o modelo perdesse a capacidade de generalizar bem, reforçando a importância de testar, avaliar e respeitar os limites do conjunto de dados utilizado.

In [ ]:
def xgb_classifier(x_train, x_test, y_train, y_test):

    # Aplicando SMOTE apenas no treino
    sm = SMOTE(random_state=42)
    x_train, y_train = sm.fit_resample(x_train, y_train)

    # Definição do classificador com parâmetros default
    xgb = XGBClassifier(random_state=42)

    # Adição do GridSearchCV
    param_grid_xgb = {
        'n_estimators': [50,100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1],
        'tree_method': ['auto'],
    }

    # Aplicação do GridSearch
    grid = GridSearchCV(xgb, param_grid_xgb, cv=5, scoring='recall')
    grid.fit(x_train, y_train)

    # Imprimindo os melhores parâmetros encontrados
    print("Melhores parâmetros encontrados:", grid.best_params_)
    
    # Previsões no conjunto de teste usando o MELHOR modelo encontrado pelo grid
    y_pred = grid.predict(x_test)

    # Apresentação das métricas obtidas:
    print("\n--- Métricas obtidas ---")

    # Obtenção do Recall
    recall = recall_score(y_test, y_pred, pos_label=1)
    print("Recall da classe 1 no teste: ", recall)
    # Obtenção do F1 Score
    f1score = f1_score(y_test, y_pred, pos_label=1)
    print("F1 Score da classe 1 no teste: ", f1score)

    # Matriz de confusão
    cm = confusion_matrix(y_test, y_pred, labels=[1,0])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Reprovado (1)", "Aprovado (0)"]
    )
    disp.plot(cmap="Blues", values_format="d")

    plt.title("Matriz de Confusão - XGBClassifier")
    plt.show()

### 2.1.1 Primeira Análise Preditiva na semana 6

&emsp; A primeira rodada de previsões é realizada ao final da Semana 6, um ponto estratégico que representa a primeira oportunidade para uma intervenção pedagógica antes que seja tarde demais. O principal desafio nesta fase é a escassez de dados de notas, incluindo dados somente até a realização do quiz 3. O objetivo, portanto, não é alcançar uma precisão absoluta, mas sim construir um modelo de **alerta precoce** que possa sinalizar os alunos que já demonstram os primeiros sinais de risco.

&emsp; A célula de código a seguir executa este processo, e a performance do modelo é detalhada através da Matriz de Confusão, que visualiza os acertos e erros, e das métricas de Recall e F1-Score, que apresentam o desempenho do modelo em sua tarefa principal: identificar corretamente os alunos do grupo "Reprovado".

In [ ]:
xgb_classifier(X1_train, X1_test, y1_train, y1_test)

#### Análise dos Resultados (Semana 6)

&emsp; Para a primeira etapa de previsão, na Semana 6, o modelo XGBoost foi otimizado via `GridSearchCV`, que indicou como melhores parâmetros `max_depth=7` e `n_estimators=200`. A performance do modelo nesta fase inicial, embora com espaço para melhorias, já aponta para a viabilidade do uso de alertas precoces.

&emsp; As métricas de avaliação, focadas na identificação da classe de reprovados (classe 1), foram as seguintes:

- **Recall da classe 1 (Reprovado):** 0.769 (76.9%)
- **F1-Score da classe 1 (Reprovado):** 0.396 (39.6%)

**Interpretação:**

&emsp; O resultado mais relevante nesta análise precoce é o **Recall de 76.9%**. Este valor indica que, de todos os alunos que efetivamente foram reprovados ao final do semestre, o modelo conseguiu identificar corretamente quase 77% deles com base nos dados disponíveis até a sexta semana. Esse é um resultado promissor, pois cumpre o objetivo primário de "capturar" a maioria dos estudantes em trajetória de risco. A matriz de confusão mostra que **20 alunos** foram corretamente sinalizados (Verdadeiros Positivos), enquanto apenas **6** em situação de risco não foram detectados (Falsos Negativos).

&emsp; No entanto, essa alta sensibilidade para encontrar os reprovados veio acompanhada de um alto número de "alarmes falsos". O modelo classificou **55 alunos** como potenciais reprovados, mas que, no fim, foram aprovados (Falsos Positivos). Essa baixa precisão é o que justifica o **F1-Score de apenas 39.6%**, que pondera o bom Recall contra o alto volume de previsões incorretas de reprovação.

&emsp; Na prática, o modelo da Semana 6 atua como um **sistema de alerta inicial de ampla cobertura**. Ele é eficaz em "lançar a rede" para encontrar os alunos em risco, mas ainda não consegue separar com precisão quem realmente precisa de uma intervenção direta.

### 2.1.2 Análise Preditiva Intermediária na Semana 8

&emsp; Avançando para a oitava semana, a análise preditiva entra em uma fase intermediária e mais robusta. Neste ponto, o objetivo é consolidar os alertas gerados na Semana 6 com um modelo mais assertivo, alimentado por um conjunto de dados mais rico que agora inclui o desempenho dos alunos até o quiz 4 e também com o resultado da primeira parcial.

&emsp; A expectativa é que, com mais informações sobre a trajetória de cada aluno, o modelo consiga não apenas confirmar as tendências de risco, mas também refinar suas previsões, aumentando a confiabilidade do diagnóstico. A célula de código a seguir executa este processo, detalhando a performance através da Matriz de Confusão e das métricas de Recall e F1-Score.

In [ ]:
xgb_classifier(X2_train, X2_test, y2_train, y2_test)

#### Análise dos Resultados (Semana 8)

&emsp; Para a análise intermediária na Semana 8, o `GridSearchCV` encontrou como melhores parâmetros `max_depth=7` e `n_estimators=200`. A performance do modelo nesta etapa, como veremos, representa uma mudança estratégica em relação à análise inicial.

&emsp; As métricas de avaliação para a classe de reprovados (classe 1) foram as seguintes:

- **Recall da classe 1 (Reprovado):** 0.538 (53.8%)
- **F1-Score da classe 1 (Reprovado):** 0.538 (53.8%)

**Interpretação:**

&emsp; O resultado da Semana 8 revela uma mudança significativa na estratégia do modelo em comparação com a análise precoce. O **Recall caiu para 53.8%**, como visto na matriz, onde o modelo identificou **14** dos 26 alunos que foram reprovados, mas deixou de sinalizar outros **12** (Falsos Negativos). Em compensação, o **F1-Score melhorou para 53.8%**, impulsionado por um grande aumento na precisão. O número de "alarmes falsos" (Falsos Positivos) foi de apenas **12**, uma redução drástica em relação à Semana 6, tornando as previsões de reprovação consideravelmente mais confiáveis.

&emsp; Esta mudança de comportamento indica que, com os dados intermediários, o modelo já consegue ser mais "criterioso". Ele deixa de ser apenas uma "rede de segurança" ampla para se tornar uma ferramenta de diagnóstico mais precisa. Este resultado sugere que a oitava semana pode representar um ponto de equilíbrio ideal entre a antecedência da previsão e a confiabilidade do alerta.

### 2.1.3 Análise Preditiva Final na Semana 12

&emsp; A análise da décima segunda semana representa o diagnóstico final do modelo preditivo. Nesta fase, com as informações até o quiz 6, o modelo dispõe do conjunto de informações mais completo até então sobre a trajetória dos alunos.

&emsp; O objetivo aqui é: verificar a capacidade máxima de acerto do modelo com os dados consolidados, para estabelecer o quão bem o classificador consegue prever o resultado final com a trajetória dos alunos quase completa. A célula a seguir executa o treinamento, teste e avaliação final do modelo.

In [ ]:
xgb_classifier(X3_train, X3_test, y3_train, y3_test)

#### Análise dos Resultados (Semana 12)

&emsp; Na análise final da Semana 12, o `GridSearchCV` ajustou o modelo para uma configuração diferente, com menor profundidade e número de estimadores (`max_depth=5`, `n_estimators=100`), indicando uma preferência por um modelo menos complexo nesta fase. A performance reflete essa mudança de forma drástica.

&emsp; As métricas de avaliação para a classe de reprovados (classe 1) foram as seguintes:

- **Recall da classe 1 (Reprovado):** 0.538 (53.8%)
- **F1-Score da classe 1 (Reprovado):** 0.596 (59.6%)

**Interpretação:**

&emsp; O resultado da Semana 12 demonstra uma mudança estratégica no comportamento do modelo, perfeitamente alinhada ao objetivo desta análise final. A **queda do Recall para 53.8%** e a **subida do F1-Score para 59.6%** indicam que o modelo abandonou a postura de "alerta" e se tornou muito mais "criterioso".

&emsp; O resultado da Semana 12 demonstra uma inversão completa na estratégia do modelo, perfeitamente alinhada ao objetivo desta análise final. A **queda do Recall para 53.8%** é confirmada pela matriz, que mostra que o modelo identificou corretamente 14 dos 26 alunos que foram reprovados, mas deixou de sinalizar outros 12 (Falsos Negativos). Em contrapartida, a **subida do F1-Score para 59.6%** é justificada por uma drástica redução nos alarmes falsos: o modelo gerou apenas **7 Falsos Positivos**, um número muito inferior ao das análises anteriores, tornando suas previsões de reprovação muito mais confiáveis.

&emsp; Este comportamento é exatamente o esperado para uma previsão final. A função do modelo aqui não é mais encontrar alunos para intervenção, mas sim atuar como uma **ferramenta de diagnóstico, confirmando com alta precisão os casos de reprovação mais evidentes**, como visto nos 14 acertos contra apenas 7 erros de classificação para este grupo. O resultado, portanto, serve como uma excelente medição da capacidade do modelo de realizar uma previsão definitiva quando a incerteza é mínima.

## 2.2 AdaBoost  

O AdaBoost (Adaptive Boosting) é um algoritmo de aprendizado supervisionado baseado na técnica de *ensemble*, que combina diversos classificadores fracos, normalmente árvores de decisão rasas, para formar um classificador forte e mais robusto. O princípio do AdaBoost é atribuir pesos maiores às instâncias que foram classificadas incorretamente em iterações anteriores, de modo que os próximos classificadores foquem nesses exemplos mais difíceis. Assim, a cada nova iteração o modelo se adapta, buscando reduzir os erros cometidos pelas árvores anteriores, até alcançar um classificador final mais preciso.  

No presente trabalho, o AdaBoost foi aplicado para prever se os alunos iriam reprovar, utilizando os mesmos conjuntos de treino e teste já balanceados com a técnica SMOTE, a fim de mitigar o desbalanceamento natural das classes. Dessa forma, garantiu-se que o modelo tivesse maior capacidade de aprender tanto com os alunos aprovados quanto com os reprovados, evitando que o algoritmo favorecesse apenas a classe majoritária.  

#### Hiperparâmetros utilizados  

- **random_state = 42**: define a semente aleatória, garantindo reprodutibilidade e comparabilidade entre execuções.  
- **n_estimators = [50, 100, 200, 300]**: representa o número de estimadores fracos (árvores) usados no processo. Valores menores tendem a gerar modelos mais rápidos, mas menos robustos, enquanto valores maiores aumentam a complexidade e podem melhorar a performance até certo ponto.  
- **learning_rate = [0.01, 0.1, 0.5, 1.0]**: controla o peso atribuído a cada estimador na combinação final. Taxas muito baixas tornam o aprendizado lento, enquanto taxas muito altas podem levar a sobreajuste. A faixa testada buscou equilibrar viés e variância.  
- **GridSearchCV(cv=5)**: a validação cruzada em 5 folds foi escolhida para garantir maior estabilidade nos resultados, testando o modelo em diferentes divisões dos dados, mas sem comprometer excessivamente o tempo de execução.  
- **scoring = 'recall'**: adotou-se o *recall* como métrica principal, priorizando a redução de falsos negativos. Esse ajuste foi fundamental, pois no contexto do projeto é mais crítico não deixar de identificar alunos em risco de reprovação do que classificar equivocadamente um aluno aprovado.  
- **n_jobs = -1**: utilização de todos os núcleos disponíveis da máquina, acelerando o processo de treinamento e validação.  

A escolha desses hiperparâmetros mostrou-se adequada para o problema proposto, permitindo ao AdaBoost explorar diferentes combinações de estimadores e taxas de aprendizado sem incorrer em sobrecarga computacional excessiva. Além disso, a priorização do *recall* como métrica de avaliação garantiu que o modelo mantivesse foco na detecção de alunos em risco de reprovação, reforçando a utilidade prática do algoritmo dentro do contexto educacional analisado.  


In [ ]:
def adaboost(x_train, x_test, y_train, y_test):

    # Aplicando SMOTE apenas no treino
    sm = SMOTE(random_state=42)
    x_train, y_train = sm.fit_resample(x_train, y_train)

    # Definição do classificador (parâmetros default)
    ab = AdaBoostClassifier(random_state=42)  # n_estimators=50, learning_rate=1.0

    # Parametros do GridSearchCV
    param_grid_ada = {
    'n_estimators': [50, 100, 200],     
    'learning_rate': [0.01, 0.1, 0.5],                      
}

    
    # Aplicação do Grid SearchCv
    grid = GridSearchCV(ab, param_grid_ada, cv=5, scoring='recall', n_jobs=-1)
    grid.fit(x_train, y_train)

    # Imprimindo os melhores parâmetros que a busca encontrou
    print("Melhores parâmetros encontrados:", grid.best_params_)

    # Previsões no conjunto de teste usando o melhor modelo
    y_pred = grid.predict(x_test)   

    # Apresentação das métricas obtidas:
    print("\n--- Métricas obtidas ---")
    # Obtenção do Recall
    recall = recall_score(y_test, y_pred, pos_label=1)
    print("Recall da classe 1 no teste: ", recall)
    # Obtenção do F1 Score
    f1score = f1_score(y_test, y_pred, pos_label=1)
    print("F1 Score da classe 1 no teste: ", f1score)

    # Matriz de confusão
    cm = confusion_matrix(y_test, y_pred, labels=[1,0])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Reprovado (1)", "Aprovado (0)"]
    )
    disp.plot(cmap="Blues", values_format="d")

    plt.title("Matriz de Confusão - AdaBoost")
    plt.show()

### 2.2.1 Análise do AdaBoost na Semana 6

&emsp; Após a avaliação do XGBoost, o modelo AdaBoost é agora testado sob as mesmas condições para a previsão precoce na Semana 6. O objetivo desta análise é comparar a abordagem de *boosting* adaptativo do AdaBoost com a de *gradient boosting* do XGBoost, utilizando o mesmo "snapshot" de dados iniciais.

&emsp; A célula de código a seguir executa o treinamento e a otimização do AdaBoost com `GridSearchCV`. A performance do modelo é avaliada com os mesmos critérios de matriz de confusão, recall e f1 score, para permitir uma comparação direta de eficácia como ferramenta de alerta precoce.

In [ ]:
adaboost(X1_train, X1_test, y1_train, y1_test)

#### Análise dos Resultados (Semana 6)

&emsp; Para a análise de alerta precoce na Semana 6, o modelo AdaBoost foi otimizado pelo `GridSearchCV`, que selecionou uma configuração de aprendizado lento e com um número baixo de estimadores (`learning_rate=0.1`, `n_estimators=50`). A performance do modelo, detalhada abaixo, mostra uma forte especialização em detectar a classe minoritária.

&emsp; As métricas de avaliação para a classe de reprovados (classe 1) foram as seguintes:

- **Recall da classe 1 (Reprovado):** 0.808 (80.8%)
- **F1-Score da classe 1 (Reprovado):** 0.339 (33.9%)

**Interpretação:**

&emsp; O modelo AdaBoost demonstrou uma performance notável em seu objetivo principal, alcançando um **Recall de 80.8%**. Este resultado, visível na matriz de confusão, significa que o modelo foi capaz de identificar corretamente **21 dos 26 alunos** que seriam reprovados, deixando apenas **5** casos passarem despercebidos (Falsos Negativos). Para uma previsão tão inicial, essa alta sensibilidade é o maior sucesso do modelo.

&emsp; No entanto, assim como observado no XGBoost, essa estratégia de alta captura veio com um custo significativo. Para garantir que poucos alunos em risco fossem ignorados, o modelo gerou um número extremamente elevado de "alarmes falsos": **77 Falsos Positivos**. Isso significa que 77 alunos que seriam aprovados foram incorretamente sinalizados como potenciais reprovados.

&emsp; A consequência direta deste comportamento é um **F1-Score muito baixo, de apenas 33.9%**. Este valor reflete o grande desequilíbrio entre o excelente Recall e uma Precisão muito baixa. Na prática, das 98 pessoas que o modelo sinalizou, apenas 21 realmente reprovaram. Portanto, o AdaBoost na Semana 6 funciona como um sistema de alerta ainda mais "agressivo" que o XGBoost, ideal para acionar intervenções de custo zero e ampla escala, mas inadequado para ações focadas e individuais.

### 2.2.2 Análise do AdaBoost na Semana 8

&emsp; Prosseguindo com a análise comparativa, o modelo AdaBoost é agora avaliado no "snapshot" de dados da Semana 8. O objetivo é observar se, com um conjunto de informações mais consolidado, o algoritmo consegue refinar suas previsões, potencialmente melhorando o equilíbrio entre a identificação de alunos em risco e a precisão de seus alertas.

&emsp; A célula de código a seguir treina e avalia o modelo para este período intermediário. A análise dos resultados irá focar na evolução do seu comportamento em relação à performance agressiva observada na Semana 6, verificando se ele se torna uma ferramenta de diagnóstico mais equilibrada.

In [ ]:
adaboost(X2_train, X2_test, y2_train, y2_test)

#### Análise dos Resultados (Semana 8)

&emsp; Na análise intermediária da Semana 8, o `GridSearchCV` optou por uma configuração de modelo mais complexa, com um número maior de estimadores (`n_estimators=200`) e uma taxa de aprendizado mais baixa (`learning_rate=0.01`). Essa mudança resultou em uma performance notavelmente diferente da fase inicial.

&emsp; As métricas de avaliação para a classe de reprovados (classe 1) foram as seguintes:

- **Recall da classe 1 (Reprovado):** 0.654 (65.4%)
- **F1-Score da classe 1 (Reprovado):** 0.459 (45.9%)

**Interpretação:**

&emsp; O modelo AdaBoost na Semana 8 demonstra uma clara evolução em sua estratégia. O **Recall diminuiu para 65.4%**, indicando que o modelo se tornou mais seletivo. A matriz de confusão mostra que ele identificou corretamente **17** alunos que foram reprovados, mas deixou de sinalizar **9** (Falsos Negativos), um número maior de "esquecidos" em comparação com a Semana 6.

&emsp; Em contrapartida, essa seletividade resultou em uma melhora expressiva do **F1-Score para 45.9%**. O motivo é a redução drástica no número de "alarmes falsos": o modelo gerou apenas **31 Falsos Positivos**, menos da metade do valor da análise anterior. Isso significa que, embora o modelo tenha deixado mais alunos em risco passarem, as suas previsões de reprovação se tornaram muito mais confiáveis e menos custosas em termos de intervenções desnecessárias.

&emsp; Assim como o XGBoost, o AdaBoost também amadureceu com mais dados, deixando de ser um alerta precoce "agressivo" para se tornar uma ferramenta de diagnóstico mais equilibrada. Suas previsões na Semana 8, embora não capturem todos os casos, são mais assertivas e direcionadas, oferecendo um balanço mais prático entre detectar o risco e evitar ações desnecessárias.

### 2.2.3 Análise Final do AdaBoost na Semana 12

&emsp; Chegando à análise final na Semana 12, o modelo AdaBoost é avaliado com o conjunto de dados mais completo, representando a trajetória praticamente consolidada dos alunos. O objetivo é mensurar a performance máxima do algoritmo e comparar seu comportamento final com o do modelo XGBoost.

&emsp; A célula de código a seguir executa o treinamento e a avaliação para este último período. A análise irá focar em determinar se o AdaBoost mantém sua estratégia equilibrada ou se, assim como o XGBoost, ele também adota uma postura mais conservadora quando a incerteza dos dados é mínima.

In [ ]:
adaboost(X3_train, X3_test, y3_train, y3_test)

#### Análise dos Resultados (Semana 12)

&emsp; Na análise final da Semana 12, o `GridSearchCV` manteve os mesmos melhores parâmetros da semana anterior (`learning_rate=0.01`, `n_estimators=200`), indicando que o modelo atingiu uma configuração estável. A performance final reflete uma pequena, porém positiva, evolução.

&emsp; As métricas de avaliação para a classe de reprovados (classe 1) foram as seguintes:

- **Recall da classe 1 (Reprovado):** 0.692 (69.2%)
- **F1-Score da classe 1 (Reprovado):** 0.493 (49.3%)

**Interpretação:**

&emsp; O modelo AdaBoost na Semana 12 apresentou uma melhora incremental em relação à sua performance intermediária. O **Recall subiu para 69.2%**, mostrando que o modelo conseguiu identificar corretamente **18 dos 26 alunos** em situação de risco, deixando **8** passarem despercebidos. Paralelamente, o **F1-Score também teve um leve aumento para 49.3%**, impulsionado por uma pequena redução no número de "alarmes falsos" para **29 Falsos Positivos**.

&emsp; Diferente do XGBoost, que mudou drasticamente sua estratégia na fase final, o AdaBoost demonstrou **consistência**. Ele manteve o perfil de um modelo equilibrado, sem sacrificar sua capacidade de detecção (Recall) para obter uma precisão extrema. O resultado é um classificador que, em sua versão final, oferece um balanço razoável entre encontrar os alunos em risco e evitar classificações incorretas.

&emsp; Conclui-se que, para este conjunto de dados, o AdaBoost se estabilizou como uma ferramenta de diagnóstico consistente nas fases mais tardias do curso, servindo como uma alternativa equilibrada, embora com performance geral inferior à do XGBoost em sua especialização final.

### 2.3 Conclusão dos Modelos Adicionais: Análise de XGBoost e AdaBoost

&emsp; A fase inicial deste projeto foi dedicada à exploração de dois algoritmos de *ensemble* de alta performance, XGBoost e AdaBoost, com o objetivo de estabelecer um benchmark de performance e compreender a complexidade do problema de prever a reprovação em diferentes estágios do semestre. Ambos os modelos foram treinados e avaliados nos marcos temporais das semanas 6, 8 e 12, revelando insights cruciais sobre a natureza da previsão.

&emsp; A análise demonstrou um padrão de comportamento fascinante e consistente em ambos os modelos. Na **Semana 6**, com dados limitados, tanto o XGBoost quanto o AdaBoost se comportaram como "sistemas de alerta precoce" agressivos, otimizando para um **alto Recall** (sensibilidade) ao custo de um número muito elevado de falsos positivos. À medida que mais dados se tornavam disponíveis nas **Semanas 8 e 12**, os modelos ajustaram sua estratégia, tornando-se mais "conservadores": o Recall diminuiu, mas a precisão e o F1-Score melhoraram, indicando uma maior confiança em suas previsões de reprovação.

&emsp; Embora ambos os modelos tenham demonstrado grande capacidade de aprendizado e adaptação, eles não foram selecionados como a solução definitiva por duas razões principais. Primeiro, a **complexidade e a natureza "caixa-preta"** desses algoritmos tornam a interpretação de suas decisões menos compreensíveis, o que é uma desvantagem em um contexto educacional onde a justificativa para uma intervenção é muito importante. Segundo, a **estratégia de previsão variável** dos modelos ao longo do tempo, mudando de foco de descobrir o máximo de reprovados possíveis para um modelo que prezava mais por manter uma previsão mais equilibrada, exigiria políticas de intervenção distintas e complexas, dificultando a implementação de uma solução padronizada.

&emsp; No entanto, a análise desses modelos foi muito importante para o projeto. Eles não apenas estabeleceram um **forte benchmark de performance** em que a solução final será comparada, mas também iluminaram a natureza fundamental do problema: a previsão precoce é uma troca entre "capturar todos os casos" e "estar certo em cada alerta".